# NB13 — Final publication figure polish

This notebook regenerates only the final publication versions of Figures 1, 4, and 8.
It does **not** retrain any model.

- Figure 1: larger correlation annotations for MDPI legibility.
- Figure 4: TabPFN and LDA-XGBoost OOF confusion matrices with adaptive annotation colors.
- Figure 8: TabPFN and LDA-XGBoost learning curves with a common y-axis.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from sklearn.metrics import confusion_matrix

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
DATASET = ROOT / '01_DATA' / 'INIAP_Dataset.xlsx'
OOF_FILE = ROOT / '03_RESULTS' / 'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE' / 'all_oof_harmonized.csv'
LC_SUMMARY = ROOT / '03_RESULTS' / 'NB12_TOP2_DIAGNOSTICS' / 'top2_learning_curve_summary.csv'
FIG = ROOT / '05_FIGURES' / 'FINAL_PUBLICATION'
FIG.mkdir(parents=True, exist_ok=True)

SEEDS = [2026, 2027, 2028]
CLASSES = ['INIAP 420', 'INIAP 425', 'INIAP 481', 'INIAP 485']

for p in [DATASET, OOF_FILE, LC_SUMMARY]:
    assert p.exists(), f'Missing required file: {p}'

print('Final figure folder:', FIG)


## Figure 1 — correlation matrix


In [ ]:
df = pd.read_excel(DATASET).rename(columns={'AspectRation':'AspectRatio'})
features = ['Area','Perimeter','MajorAxisLength','MinorAxisLength','AspectRatio','ConvexArea','EquivDiameter','Extent','Solidity','roundness','Compactness']
labels = ['Area','Perimeter','Major axis','Minor axis','Aspect ratio','Convex area','Equiv. diameter','Extent','Solidity','Roundness','Compactness']
corr = df[features].corr(method='pearson').to_numpy()

fig, ax = plt.subplots(figsize=(10.8, 9.3))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm', aspect='equal')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', rotation_mode='anchor', fontsize=10.5)
ax.set_yticklabels(labels, fontsize=10.5)

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        value = corr[i, j]
        color = 'white' if abs(value) >= 0.55 else 'black'
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=9.5, fontweight='medium', color=color)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Pearson correlation (r)', fontsize=11)
cbar.ax.tick_params(labelsize=10)
ax.set_xlabel('')
ax.set_ylabel('')
fig.tight_layout()

for ext in ['png','pdf','svg']:
    path = FIG / f'Figure1_Correlation_Matrix_PUBLICATION.{ext}'
    if ext == 'png':
        fig.savefig(path, dpi=600, bbox_inches='tight')
    else:
        fig.savefig(path, bbox_inches='tight')
plt.show()


## Figure 4 — Top-2 confusion matrices


In [ ]:
oof = pd.read_csv(OOF_FILE)
models = ['TabPFN', 'LDA_XGBoost']
panel_letters = ['A', 'B']
seed_matrices = {}

for model in models:
    mats = []
    g = oof[(oof['feature_set'] == 'FULL_11') & (oof['model'] == model)].copy()
    assert len(g) == 12000, (model, len(g))
    for seed in SEEDS:
        gs = g[g['seed'] == seed]
        assert len(gs) == 4000
        assert gs['row_id'].nunique() == 4000
        cm = confusion_matrix(gs['y_true'], gs['y_pred'], labels=CLASSES, normalize='true')
        mats.append(cm)
    seed_matrices[model] = np.stack(mats, axis=0)

mean_matrices = {m: seed_matrices[m].mean(axis=0) for m in models}
fig, axes = plt.subplots(1, 2, figsize=(11.6, 5.2), constrained_layout=True)
images=[]

for ax, model, letter in zip(axes, models, panel_letters):
    mat = mean_matrices[model]
    im = ax.imshow(mat, vmin=0, vmax=1, cmap='Blues')
    images.append(im)
    ax.set_xticks(range(len(CLASSES)))
    ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES, rotation=35, ha='right', fontsize=10)
    ax.set_yticklabels(CLASSES, fontsize=10)
    ax.set_xlabel('Predicted cultivar', fontsize=11)
    ax.set_ylabel('True cultivar', fontsize=11)
    ax.text(0.02, 0.98, letter, transform=ax.transAxes, ha='left', va='top', fontsize=15,
            fontweight='bold', color='white',
            path_effects=[pe.withStroke(linewidth=2.5, foreground='black')])
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            value = mat[i,j]
            ax.text(j, i, f'{value:.3f}', ha='center', va='center', fontsize=10,
                    fontweight='bold', color=('white' if value >= 0.50 else 'black'))

cbar = fig.colorbar(images[0], ax=axes, fraction=0.025, pad=0.03)
cbar.set_label('Mean row-normalized proportion', fontsize=10)
cbar.ax.tick_params(labelsize=9)

for ext in ['png','pdf','svg']:
    path = FIG / f'Figure4_Confusion_Matrices_TabPFN_LDAXGBoost_PUBLICATION.{ext}'
    if ext == 'png':
        fig.savefig(path, dpi=600, bbox_inches='tight')
    else:
        fig.savefig(path, bbox_inches='tight')
plt.show()


## Figure 8 — Top-2 learning curves with common y-axis


In [ ]:
s = pd.read_csv(LC_SUMMARY)
models = [('TabPFN','A'), ('LDA_XGBoost','B')]
fig, axes = plt.subplots(1, 2, figsize=(11.8, 5.0), sharey=True, constrained_layout=True)

for ax, (model, letter) in zip(axes, models):
    g = s[s['model'] == model].sort_values('mean_n_train')
    ax.errorbar(g['mean_n_train'], g['mean_train_macro_f1'], yerr=g['sd_train_macro_f1'],
                marker='o', capsize=4, linewidth=1.8, label='Training macro-F1')
    ax.errorbar(g['mean_n_train'], g['mean_test_macro_f1'], yerr=g['sd_test_macro_f1'],
                marker='o', capsize=4, linewidth=1.8, label='Outer-test macro-F1')
    ax.set_xlim(550, 3290)
    ax.set_ylim(0.960, 1.002)
    ax.set_xticks(g['mean_n_train'])
    ax.set_xlabel('Training samples', fontsize=11)
    ax.grid(True, alpha=0.18)
    ax.legend(loc='lower right', fontsize=8)
    ax.text(0.02, 0.98, letter, transform=ax.transAxes, ha='left', va='top', fontsize=14, fontweight='bold')
    ax.tick_params(labelsize=9)

axes[0].set_ylabel('Macro-F1', fontsize=11)

for ext in ['png','pdf','svg']:
    path = FIG / f'Figure8_Learning_Curves_TabPFN_LDAXGBoost_PUBLICATION.{ext}'
    if ext == 'png':
        fig.savefig(path, dpi=600, bbox_inches='tight')
    else:
        fig.savefig(path, bbox_inches='tight')
plt.show()


## Manuscript captions

**Figure 4.** Mean row-normalized outer-test out-of-fold confusion matrices across the three repeated seeds for the two leading models using the full 11-descriptor representation: **(A)** TabPFN and **(B)** LDA-XGBoost. For each seed, every grain contributed exactly one outer-test prediction; seed-specific normalized matrices were then averaged, so repeated seeds were not treated as independent grains.

**Figure 8.** Leakage-safe diagnostic learning curves for the two leading models using the full 11-descriptor representation: **(A)** TabPFN and **(B)** LDA-XGBoost. Curves show mean training and untouched outer-test macro-F1 across the 15 repeated outer-fold evaluations, with error bars representing the standard deviation. Training fractions were sampled only from the corresponding outer-training fold; the outer-test fold remained unchanged at every training size.
